In [1]:
import os
import gymnasium as gym
from sb3_contrib.ppo_mask import MaskablePPO, MlpPolicy as MLP_PPO

from netsim.netSimPy import *
from netsim.netSimPy.common.callbacks import MyBestCallback
from netsim.gym_basic.envs import MASKING_RMSA_ENV
from stable_baselines3.common.monitor import Monitor
import tensorflow as tf
import logging

logging.getLogger("tensorflow").setLevel(logging.FATAL)
from sb3_contrib.common.maskable.evaluation import (
    evaluate_policy as evaluate_policy_masking,
)

tf.__version__

/Users/jbcedeno/Documents/projcts/multiband-gymnasium/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'2.16.2'

### Training:


In [ ]:
# FILES AND DIRECTORIES:
__file__ = "ACTION_RMSA_ENV.ipynb"
absolutepath = os.path.abspath(__file__)
file_name = os.path.basename(os.path.abspath(__file__)).split(".")[0]
networkPaths = "/Users/jbcedeno/Documents/projcts/multiband-gymnasium/networks/nsfnet"
log_dir = f"./tmp/{file_name}/"
os.makedirs(log_dir, exist_ok=True)
tensorboard_log = f"./tb/{file_name}/nsfnet/"

# SIMULATION PARAMS:
# TOTAL_TIMESTEPS = int(4e6)
TOTAL_TIMESTEPS = int(8e6)
EPISODE_LENGTH = 1000
N_EVALUATION_EPISODES = 5
EVAL_FREQ = N_EVALUATION_EPISODES * EPISODE_LENGTH
N_BLOCKS = 5
N_PATHS = 3
M_LAMBDA = 30000  # 300 Erlangs

# Building the network simulator:
network = Network(
    networkFileName=networkPaths + "/network.json",
    pathsFileName=networkPaths + "/routes.json",
    bitrateFilename=networkPaths + "/bitrates_c_bands.json",
)
generator = EventsGenerator(mLambda=M_LAMBDA)
sim_args = dict(eventsGenerator=generator, network=network)
simulator = NetworkSimulator(**sim_args)


# Building the environment:
env_args = dict(
    simulator=simulator,
    episode_length=EPISODE_LENGTH,
    j=N_BLOCKS,
    n_paths=N_PATHS,
)

env: MASKING_RMSA_ENV = Monitor(gym.make("MASKING_RMSA_ENV-v0", **env_args), log_dir)

# MODEL HYPERPARAMETERS:
mArgs = dict(
    learning_rate=0.001,
    clip_range=0.05,
    n_steps=2048,
    gamma=0.9,
    gae_lambda=0.92,
    ent_coef=0.002,
    n_epochs=16,
)
model = MaskablePPO(
    MLP_PPO, env, verbose=0, seed=3, tensorboard_log=tensorboard_log, **mArgs
)

eval_env: MASKING_RMSA_ENV = Monitor(
    gym.make("MASKING_RMSA_ENV-v0", **env_args), log_dir + "eval_env"
)
# best_model_save_path = os.path.join(log_dir, "best_model")
callback = MyBestCallback(
    eval_env,
    n_eval_episodes=N_EVALUATION_EPISODES,
    eval_freq=EVAL_FREQ,
    log_path=log_dir,
    best_model_save_path=log_dir,
    verbose=1,
    use_masking=True,
)
# train_model = model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=callback)


# 950.35
# 971.20

./tmp/ACTION_RMSA_ENV/


In [3]:
# Total blocked events: 1 for a BP=0.0
# Total blocked events: 19 for a BP=0.0001
# Total blocked events: 208 for a BP=0.001
# Total blocked events: 892 for a BP=0.0045
# Total blocked events: 2358 for a BP=0.0118
# Total blocked events: 4427 for a BP=0.0221
# Total blocked events: 6749 for a BP=0.0337
# Total blocked events: 8863 for a BP=0.0443

In [ ]:
# loads = [10000, 15000, 20000, 25000, 30000, 35000, 40000, 45000]
loads = [10000, 12500, 15000, 17500, 20000, 22500, 25000]
n_episodes = 2000

In [5]:
from netsim.netSimPy.common.allocators import sap_ff
from netsim.netSimPy.common.evaluators import SimpleEvaluator

for l in loads:
    network = Network(
        networkFileName=networkPaths + "/network.json",
        pathsFileName=networkPaths + "/routes.json",
        bitrateFilename=networkPaths + "/bitrates_c_bands.json",
    )
    generator = EventsGenerator(mLambda=l)

    sim_args = dict(
        eventsGenerator=generator,
        network=network,
        allocator=sap_ff(3, ["C"]),
    )

    simulator = NetworkSimulator(**sim_args)

    simulator.run(n_episodes * EPISODE_LENGTH, SimpleEvaluator())

Total blocked events: 2 for a BP=1e-06
Total blocked events: 21 for a BP=1.05e-05
Total blocked events: 167 for a BP=8.35e-05
Total blocked events: 683 for a BP=0.0003415
Total blocked events: 2031 for a BP=0.0010155
Total blocked events: 4605 for a BP=0.0023025
Total blocked events: 8906 for a BP=0.004453


In [6]:
bp_model = []
for t in loads:
    network = Network(
        networkFileName=networkPaths + "/network.json",
        pathsFileName=networkPaths + "/routes.json",
        bitrateFilename=networkPaths + "/bitrates_c_bands.json",
    )
    generator = EventsGenerator(mLambda=t)
    sim_args = dict(eventsGenerator=generator, network=network)
    simulator = NetworkSimulator(**sim_args)

    # Building the environment:
    env_args = dict(
        simulator=simulator,
        episode_length=EPISODE_LENGTH,
        j=N_BLOCKS,
        n_paths=N_PATHS,
    )

    env: MASKING_RMSA_ENV = Monitor(gym.make("MASKING_RMSA_ENV-v0", **env_args), log_dir)

    smooth_model = MaskablePPO.load(f"./tmp/{file_name}/best_smooth_model")

    mean_reward, _ = evaluate_policy_masking(
        smooth_model,
        env,
        n_eval_episodes=n_episodes,
        deterministic=True,
        use_masking=True,
    )
    print(mean_reward)
    bp_model.append(mean_reward)

/Users/jbcedeno/Documents/projcts/multiband-gymnasium/.venv/lib/python3.10/site-packages/gymnasium/utils/passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")
/Users/jbcedeno/Documents/projcts/multiband-gymnasium/.venv/lib/python3.10/site-packages/gymnasium/utils/passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `step()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")


1000.0
999.99
999.948
999.674
998.643
996.162
991.35
